# 04 — Covariance shrinkage: does a better Σ help the optimizers?

Ledoit–Wolf (registered) and OAS (sensitivity) replace the sample covariance inside GMV, MaxSharpe and BL(0.1); the optimizers themselves do not change. Each run is scored by the rolling backtest against EW and 60/40. Hypotheses are fixed in §6; cross-model tests are in notebooks 08–09. Run labels: (S) sample, (LW) Ledoit–Wolf, (OAS) OAS; MS is MaxSharpe.

## §1 Shrinkage and the idea

With 252 days and 13 assets ($T/N \approx 19$) the sample covariance is noisy, and optimizers that invert it are most sensitive along its smallest eigenvalues. Shrinkage blends the sample matrix with a structured target, trading some bias for less estimation error; both estimators here shrink toward a scaled identity. A long-only constraint already acts like shrinkage on minimum-variance weights, so the effect may be small.

How much each optimizer should respond depends on how much it relies on Σ rather than μ:

- **GMV:** no μ; Σ is the only input, so shrinkage should matter most.
- **MaxSharpe:** the raw 252-day sample mean; its noise should swamp any gain from a better Σ.
- **BL(0.1):** μ is already shrunk toward the EW-implied prior, so the estimator should barely matter.

## §2 The estimators

**General form.** A shrinkage estimator blends the sample covariance $S$ with a target $F$:

$$\hat\Sigma(\delta) = (1-\delta)\,S + \delta\,F, \qquad 0 \le \delta \le 1 .$$

**Ledoit–Wolf.** For the scaled identity $F = mI$, $m = \mathrm{tr}(S)/N$, Ledoit–Wolf choose the $\delta$ that minimizes expected squared Frobenius loss and estimate it in closed form:

$$\hat\delta = \min\!\left(1,\; \frac{\bar b^2}{d^2}\right), \qquad d^2 = \lVert S - mI \rVert^2, \qquad \bar b^2 = \frac{1}{T^2} \sum_{t=1}^{T} \lVert x_t x_t' - S \rVert^2 ,$$

with $x_t$ the demeaned daily return vector and $\lVert A \rVert^2 = \mathrm{tr}(AA')/N$. Eigenvectors are unchanged and each eigenvalue moves toward $m$: $\lambda_i \to (1-\delta)\lambda_i + \delta m$.

**OAS.** Same target. OAS chooses $\delta$ to approximate the oracle (minimum mean squared error) shrinkage under Gaussian returns, again in closed form in $T$, $N$, $\mathrm{tr}(S)$ and $\mathrm{tr}(S^2)$.

**In the classes.** `ledoit_wolf_cov` and `oas_cov` (`src/maplab/covariance.py`) wrap `sklearn.covariance` on the trailing 252 daily log returns ($S$ with ddof 0) and annualize by 252; `sample_cov` uses ddof 1. A strategy takes the estimator as `cov_estimator=` and uses it for Σ inside `_estimate` (defined in notebook 01 §3); nothing else changes. GMV and BL(0.1) take only Σ from `_estimate`. MaxSharpe also optimizes on μ = mean·252 + ½diag(Σ), so the estimator reaches its μ through the Itô term. Two design checks, run at the three snapshot dates in `tests/test_nb04_checks.py`, both hold: `BlackLitterman` with $k = 0$ returns EW under every estimator, because the prior $\Pi = \delta\Sigma w_{EW}$ moves with Σ; and recomputing MaxSharpe with ½diag of the sample Σ instead stays inside the pre-set limits (10 bp/yr on μ, 1% on weights), so `_estimate` is left as is.

## Setup

Prices, log returns for estimation, simple returns for the backtest, BIL as $r_f$, and the panel the classes read.

In [ ]:
# Colab only: get the repo and install it. Does nothing when run locally.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    repo = "/content/multi-asset-portfolio-lab"
    if not os.path.exists(repo):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/FranQuant/multi-asset-portfolio-lab", repo], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", repo], check=True)
    sys.path.insert(0, f"{repo}/src")
    os.chdir(f"{repo}/notebooks")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.covariance import LedoitWolf, OAS, empirical_covariance

import maplab as ml
from maplab import Panel, apply_style, GMV, MaxSharpe, BlackLitterman, EqualWeight, FixedWeight, backtest

from helpers import nb04 as h
from helpers import common as cm

In [ ]:
apply_style()

prices = ml.load_prices()
log_returns = ml.to_log_returns(prices)        # for estimation (mu, Sigma)
simple_returns = ml.to_simple_returns(prices)  # for compounding the backtest
rf_daily = ml.load_rf_returns()["BIL"]
panel = Panel({"prices": prices, "returns": log_returns, "rf": ml.load_rf_returns()})
split_ts = pd.Timestamp(ml.TRAIN_TEST_SPLIT)

At 2022-12-31: sklearn's shrinkage weight δ for each estimator, and a check that each equals $(1-\delta)S + \delta mI$ as in §2.

In [ ]:
asof_split = pd.Timestamp("2022-12-31")
rets_split = panel.slice(asof_split, "returns", ml.COV_LOOKBACK)[ml.UNIVERSE]
n_split = len(rets_split)
assert n_split == ml.COV_LOOKBACK, f"expected {ml.COV_LOOKBACK} rows, got {n_split}"

S = empirical_covariance(rets_split.to_numpy())
S_ddof = S * ml.TRADING_DAYS * n_split / (n_split - 1)
S_sample = ml.sample_cov(rets_split).to_numpy()
ddof_gap = np.max(np.abs(S_ddof - S_sample))

m = np.trace(S) / len(ml.UNIVERSE)

lw_fit = LedoitWolf().fit(rets_split.to_numpy())
delta_lw_split = lw_fit.shrinkage_
target_lw = (1 - delta_lw_split) * S + delta_lw_split * m * np.eye(len(ml.UNIVERSE))
lhs_lw = ml.ledoit_wolf_cov(rets_split).to_numpy() / ml.TRADING_DAYS
lw_gap = np.max(np.abs(lhs_lw - target_lw))

oas_fit = OAS().fit(rets_split.to_numpy())
delta_oas_split = oas_fit.shrinkage_
target_oas = (1 - delta_oas_split) * S + delta_oas_split * m * np.eye(len(ml.UNIVERSE))
lhs_oas = ml.oas_cov(rets_split).to_numpy() / ml.TRADING_DAYS
oas_gap = np.max(np.abs(lhs_oas - target_oas))

print(f"2022-12-31: δ_LW {delta_lw_split:.4f}, δ_OAS {delta_oas_split:.4f}; "
      f"both equal (1−δ)S + δmI to {max(lw_gap, oas_gap):.1e}; "
      f"sample_cov = 252·T/(T−1)·S to {ddof_gap:.1e}")

## §3 Shrinkage dose through time

At each of the 208 rebalances: sklearn's $\delta$ for LW and OAS, and the condition number (largest over smallest eigenvalue: the higher it is, the more inverting Σ amplifies noise) of $\Sigma_{sample}$ and $\Sigma_{LW}$.

In [ ]:
rdates = ml.rebalance_dates(log_returns.index)
rdates = rdates[rdates >= ml.first_eligible_rebalance(log_returns.index)]
assert len(rdates) == 208, f"expected 208 rebalance labels, got {len(rdates)}"

dose = h.dose_table(panel, rdates)
assert len(dose) == 208

dose_stats = dose.describe(percentiles=[.25, .5, .75]).loc[["min", "25%", "50%", "75%", "max"]]
dose_stats.columns = ["δ LW", "δ OAS", "cond(Σ sample)", "cond(Σ LW)"]
pd.concat([ml.reporting.show_table(dose_stats[["δ LW", "δ OAS"]], dp=3),
           ml.reporting.show_table(dose_stats[["cond(Σ sample)", "cond(Σ LW)"]], dp=1)], axis=1)

In [ ]:
fig = h.dose_figure(dose, split_ts)
plt.show()

**Reading.** Shrinkage is light: the median $\delta_{LW}$ is 0.037 (range 0.020–0.134) and OAS shrinks less (median 0.021). Even so, it cuts the median condition number from 512.7 to 119.9, so $\Sigma_{LW}$ is far easier to invert than the sample matrix. If a better Σ helps anywhere, it should be in GMV, which uses nothing else.

At 2022-12-31: the spectrum before and after LW. The smallest eigenvalues rise most in relative terms, which lowers the condition number; those are the directions along which $\Sigma^{-1}$, and so GMV and MaxSharpe weights, are most sensitive.

In [ ]:
fig = h.eigen_scree(S, delta_lw_split, m)
plt.show()

**Reading.** At 2022-12-31 the three largest eigenvalues barely move while the smallest more than doubles, which is how a δ of 0.0319 cuts the condition number from 299.5 to 122.4 (legend).

## §4 Where shrinkage acts

At the three snapshot dates: the share of $\Delta = \Sigma_{LW} - \Sigma_{sample}$ on the diagonal, $\lVert \mathrm{diag}(\Delta) \rVert_F^2 / \lVert \Delta \rVert_F^2$.

In [ ]:
DATES = [pd.Timestamp(d) for d in ["2014-12-31", "2017-12-31", "2022-12-31"]]

diag_share_table, _ = h.diag_share_tables(panel, DATES)

diag_shown = diag_share_table.rename(columns={"diag_share": "diagonal share (%)",
                                              "off_diag_share": "off-diagonal share (%)"}).rename_axis(None)
ml.reporting.show_table(diag_shown, pct=list(diag_shown.columns), pct_dp=1)

**Reading.** Most of the change is in the correlations, not the variances: the diagonal share is 27.3%, 33.3% and 12.9%, so 67–87% of the move is off-diagonal. Covariances all shrink by the factor $1-\delta$ while variances move toward $m$, so every correlation shrinks toward 0 by $\rho_{LW}/\rho_S = (1-\delta)/\sqrt{a_i a_j}$, with $a_i = 1-\delta+\delta m/s_{ii}$; the ratio is smallest for low-variance assets. At 2022-12-31 UUP has the lowest sample vol (9.27%, notebook 01 §4.1), so its correlations, the hedge behind the UUP corner, shrink proportionally more than any other asset's; the heatmaps show sample, LW and their difference at that date.

In [ ]:
fig = h.corr_heatmaps(S_sample, lhs_lw * ml.TRADING_DAYS)
plt.show()

### `lw_corr` — correlations only

A snapshot-only variant: LW applied to the correlation matrix, sample vols kept. Never backtested. It separates the correlation part of LW's weight change from the vol part.

In [ ]:
lw_corr_table = h.lw_corr_table(panel, DATES)

lw_corr_shown = lw_corr_table.rename(columns={"halfL1_LW_vs_sample_%": "½L1 LW vs sample (%)",
                                              "halfL1_lwcorr_vs_sample_%": "½L1 corr-only vs sample (%)"})
ml.reporting.show_table(lw_corr_shown, dp=1)

**Reading.** Shrinking correlations alone moves weights by 0.07–1.64× the full-LW move, and by less than full LW in 8 of 9 cases. So although most of the change to the matrix is off-diagonal (§4), the vol part of LW also moves weights.

## §5 What changes in the weights

How far LW and OAS move each optimizer's weights away from the sample-Σ weights at the three dates (½L1, %), then the largest holding at 2022-12-31.

In [ ]:
weight_table, _ = h.weight_tables(panel, DATES)

halfl1 = weight_table["halfL1_vs_sample_%"].drop("sample", level="estimator").unstack(["strategy", "estimator"])
cols = [(s, e) for s in ["GMV", "MaxSharpe", "BL(0.1)"] for e in ["ledoit_wolf", "oas"]]
halfl1 = halfl1[cols]
halfl1.columns = [f"{s} {'LW' if e == 'ledoit_wolf' else 'OAS'}" for s, e in cols]
display(ml.reporting.show_table(halfl1.rename_axis(None), dp=1))

last = weight_table.xs(weight_table.index.get_level_values("asof").max(), level="asof")
print("largest weight at 2022-12-31: " + "; ".join(
    f"{s} {dict(sample='S', ledoit_wolf='LW', oas='OAS')[e]}: {r['argmax']} {r['max_w_%']:.2f}%" for (s, e), r in last.loc[["GMV", "MaxSharpe"]].iterrows()))

**Reading.** BL(0.1) barely moves, as §1 expected; GMV and MaxSharpe move more, by similar amounts. Across the three dates LW moves GMV by 4.8–11.7% of the portfolio (½L1: half the sum of absolute weight changes, the share that would have to trade), MaxSharpe by 3.2–10.5% and BL(0.1) by 1.5–2.2%; OAS moves weights by at most 9.5%.

**Reading.** The 2022 corner is not estimation noise that shrinkage removes: UUP is the largest weight in all six, GMV 58.12% sample, 55.35% LW, 56.36% OAS; MaxSharpe 74.41%, 74.09%, 74.27%. Shrinkage trims it by at most 2.8 pp.

## §6 Hypotheses

Hypotheses fixed after exploratory runs and transcribed into `registrations/nb04.toml` (at 210adef), then evaluated as written. Each test uses the paired daily net-return difference (LW run − sample run) over the full window, $t = \bar d / s_d \cdot \sqrt{n}$; the default expectation for all three is a null.

| | Hypothesis | Counts if |
|---|---|---|
| H1 (GMV) | GMV(LW) − GMV(S) mean net return ≥ 0 | t ≥ 2, read together with the §9 β decomposition |
| H2 (MaxSharpe) | MaxSharpe(LW) − MaxSharpe(S) mean net return ≤ 0 | t ≤ −2 |
| BL(0.1) | LW − sample indistinguishable from zero (predicted null) | rejected only if t is beyond ±2 |

## §7 Out-of-sample results

Nine runs (three optimizers × three estimators), EW and 60/40, walk-forward as in notebook 01 §5. The sample runs, EW and 60/40 are the same as in notebooks 01 and 03; `tests/test_reproduction.py` checks their Sharpe ratios.

In [ ]:
strategies = {
    "GMV(S)": GMV(cov_estimator=ml.sample_cov),
    "GMV(LW)": GMV(cov_estimator=ml.ledoit_wolf_cov),
    "GMV(OAS)": GMV(cov_estimator=ml.oas_cov),
    "MS(S)": MaxSharpe(cov_estimator=ml.sample_cov),
    "MS(LW)": MaxSharpe(cov_estimator=ml.ledoit_wolf_cov),
    "MS(OAS)": MaxSharpe(cov_estimator=ml.oas_cov),
    "BL(S)": BlackLitterman(cov_estimator=ml.sample_cov, k=0.1),
    "BL(LW)": BlackLitterman(cov_estimator=ml.ledoit_wolf_cov, k=0.1),
    "BL(OAS)": BlackLitterman(cov_estimator=ml.oas_cov, k=0.1),
    "EW": EqualWeight(),
    "60/40": FixedWeight({"SPY": 0.6, "IEF": 0.4}, name="60/40"),
}

results = {}
for name, strat in strategies.items():
    net, wlog, diag = backtest(strat, simple_returns, panel)
    results[name] = {"net": net, "wlog": wlog, "diag": diag, "strat": strat}

In [ ]:
summary_table = cm.summary_by_window(results, split_ts, rf_daily, cost_drag=True)

In [ ]:
ms_bl = [results[n]["strat"] for n in ["MS(S)", "MS(LW)", "MS(OAS)", "BL(S)", "BL(LW)", "BL(OAS)"]]
n_fallback = sum(len(s.fallback_dates) for s in ms_bl)
n_retry = sum(len(s.retry_dates) for s in ms_bl)
n_conc = {e: len(results[f"BL({e})"]["strat"].concentration_dates) for e in ["S", "LW", "OAS"]}
print(f"MaxSharpe and BL: {n_fallback} fallbacks, {n_retry} retries; "
      f"BL concentrated (effective N < 5): S {n_conc['S']}, LW {n_conc['LW']}, OAS {n_conc['OAS']} of {len(rdates)}")

In [ ]:
cm.show_overview(cm.overview_table(summary_table, order=list(results)))

**Reading.** Only GMV gains visibly: LW lifts its full-window Sharpe from 0.62 to 0.70, while MaxSharpe (0.72 → 0.73) and BL(0.1) (0.64 → 0.64) barely change; LW lowers turnover for all three. No shrunk run reaches 60/40's 0.91: the best, MaxSharpe(OAS), has 0.73, and GMV(LW) stays below EW's 0.74.

## §8 Paired differences

For each optimizer, LW − sample (registered) and OAS − sample (sensitivity): the daily net-return difference $d$, its annualized mean, tracking error (TE, the volatility of $d$) and $t = \bar d / s_d \cdot \sqrt{n}$, for the full and test windows; the line below gives GMV's cost-drag difference and gross-of-costs mean.

In [ ]:
pairs = [
    ("GMV(LW)", "GMV(S)", "registered"), ("GMV(OAS)", "GMV(S)", "sensitivity"),
    ("MS(LW)", "MS(S)", "registered"), ("MS(OAS)", "MS(S)", "sensitivity"),
    ("BL(LW)", "BL(S)", "registered"), ("BL(OAS)", "BL(S)", "sensitivity"),
]

paired_rows = []
paired_series = {}
for est_name, base_name, kind in pairs:
    net_est = results[est_name]["net"]
    net_base = results[base_name]["net"]
    idx = net_est.index.intersection(net_base.index)
    diff = net_est.loc[idx] - net_base.loc[idx]
    paired_series[(est_name, base_name)] = diff
    win_defs = {
        "full": diff.index >= diff.index.min(),
        "train (<= split)": diff.index <= split_ts,
        "test (> split)": diff.index > split_ts,
    }
    for window_name, sel in win_defs.items():
        d = diff.loc[sel]
        ann_mean = float(d.mean() * ml.TRADING_DAYS)
        te = float(d.std(ddof=1) * np.sqrt(ml.TRADING_DAYS))
        t_stat = float(d.mean() / (d.std(ddof=1) / np.sqrt(len(d))))
        cost_drag_est = float(summary_table.loc[(est_name, window_name), "cost_drag"])
        cost_drag_base = float(summary_table.loc[(base_name, window_name), "cost_drag"])
        cost_drag_diff = cost_drag_est - cost_drag_base
        gross_diff = ann_mean + cost_drag_diff
        paired_rows.append({
            "pair": f"{est_name} - {base_name}", "kind": kind, "window": window_name,
            "ann_mean_diff": ann_mean, "TE": te, "t": t_stat,
            "cost_drag_diff": cost_drag_diff, "gross_of_costs_mean_diff": gross_diff,
        })

paired_table = pd.DataFrame(paired_rows).set_index(["pair", "kind", "window"])

paired_shown = (paired_table.loc[paired_table.index.get_level_values("window") != "train (<= split)",
                                 ["ann_mean_diff", "TE", "t"]]
                .rename(columns={"ann_mean_diff": "mean diff (%/yr)", "TE": "TE (%/yr)"}))
display(ml.reporting.show_table(paired_shown, pct=["mean diff (%/yr)", "TE (%/yr)"], pct_dp=2, dp=2))

g = paired_table.loc[("GMV(LW) - GMV(S)", "registered", "full")]
fmt = lambda v: f"{v:+.2%}".replace("-", "−")
print(f"GMV(LW) − GMV(S), full window: cost-drag difference {fmt(g['cost_drag_diff'])}/yr, "
      f"gross-of-costs mean {fmt(g['gross_of_costs_mean_diff'])}/yr")

**Reading.** Only GMV's gain clears t = 2, and only over the full window: GMV(LW) − GMV(S) is +0.25%/yr (t 2.34) full and +0.17%/yr (t 0.94) in the test window, and costs explain almost none of it (cost-drag difference −0.02%/yr). MaxSharpe gains +0.40%/yr (t 1.61) and BL(0.1) +0.07%/yr (t 1.00); OAS − sample gives t 1.14, 1.53 and −0.05.

In [ ]:
fig = h.cumulative_lw_figure(paired_series, split_ts)
plt.show()

In [ ]:
fig = h.halfl1_lw_figure(results, split_ts)
plt.show()

**Reading.** Most of GMV's gain from LW accrued by 2021, part of it in the March 2020 crash; some was given back in 2022 and the line is roughly flat since, consistent with the test-window t of 0.94. MaxSharpe's gain comes in jumps, and its allocation distance from the sample run spikes to 30–40% in 2020–21; BL(0.1)'s stays mostly below 10%.

## §9 $\beta$ decomposition

Plain OLS (no HAC) of each run's daily excess return on the market's. Each LW − sample difference splits into $\Delta\alpha$ and a $\beta$ part, $\Delta\beta$ × mean market excess return.

In [ ]:
MKT_simple = (simple_returns["SPY"] - rf_daily).dropna()

capm_table = ml.inference.capm_table(results, ["GMV(S)", "GMV(LW)", "MS(S)", "MS(LW)", "BL(S)", "BL(LW)"],
                                     MKT_simple, rf_daily)

capm_shown = capm_table.rename(columns={"alpha_ann": "α (%/yr)", "t_alpha": "t(α)", "beta": "β", "r2": "R²"}).rename_axis(None)
ml.reporting.show_table(capm_shown, pct=["α (%/yr)"], pct_dp=2, dp=2, int_cols=["n"])

In [ ]:
decomp_rows = {}
for est_name, base_name, label in [("GMV(LW)", "GMV(S)", "GMV"), ("MS(LW)", "MS(S)", "MaxSharpe"), ("BL(LW)", "BL(S)", "BL(0.1)")]:
    alpha_base = float(capm_table.loc[base_name, "alpha_ann"])
    beta_base = float(capm_table.loc[base_name, "beta"])
    alpha_est = float(capm_table.loc[est_name, "alpha_ann"])
    beta_est = float(capm_table.loc[est_name, "beta"])

    net_est = results[est_name]["net"]
    idx = net_est.index.intersection(rf_daily.index).intersection(MKT_simple.index)
    mkt_ann_mean = float(MKT_simple.loc[idx].mean() * ml.TRADING_DAYS)

    d_alpha = alpha_est - alpha_base
    d_beta = beta_est - beta_base
    beta_part = d_beta * mkt_ann_mean

    realized_diff_full = float(paired_table.loc[(f"{est_name} - {base_name}", slice(None), "full"), "ann_mean_diff"].iloc[0])
    decomp_rows[label] = {"Δα (%/yr)": d_alpha, "Δβ": d_beta, "β part (%/yr)": beta_part,
                          "difference (%/yr)": realized_diff_full}

decomp_shown = pd.DataFrame(decomp_rows).T
ml.reporting.show_table(decomp_shown, pct=["Δα (%/yr)", "β part (%/yr)", "difference (%/yr)"], pct_dp=2, dp=4)

**Reading.** GMV's gain is mostly α, not a shift in market β: $\Delta\alpha$ +0.16%/yr and $\beta$ part +0.10%/yr of the +0.25%/yr difference. MaxSharpe: $\Delta\alpha$ +0.28%/yr, $\beta$ part +0.12%/yr.

## Verdict table

The §6 rules applied mechanically to the statistics above; `[applied: …]` states how an ambiguous rule was read.

In [ ]:
reg = ml.inference.load_registration("nb04")

t_full_gmv = float(paired_table.loc[("GMV(LW) - GMV(S)", "registered", "full"), "t"])
t_test_gmv = float(paired_table.loc[("GMV(LW) - GMV(S)", "registered", "test (> split)"), "t"])
t_full_ms = float(paired_table.loc[("MS(LW) - MS(S)", "registered", "full"), "t"])
t_test_ms = float(paired_table.loc[("MS(LW) - MS(S)", "registered", "test (> split)"), "t"])
mean_full_ms = float(paired_table.loc[("MS(LW) - MS(S)", "registered", "full"), "ann_mean_diff"])
t_full_bl = float(paired_table.loc[("BL(LW) - BL(S)", "registered", "full"), "t"])
t_test_bl = float(paired_table.loc[("BL(LW) - BL(S)", "registered", "test (> split)"), "t"])

stats = {
    "H1": dict(t_full=t_full_gmv, display=f"t_full={t_full_gmv:.2f} (test t={t_test_gmv:.2f}, descriptive)"),
    "H2": dict(t_full=t_full_ms, display=f"t_full={t_full_ms:.2f} (test t={t_test_ms:.2f}, descriptive), "
                                         f"full mean sign={'+' if mean_full_ms > 0 else '−'}"),
    "BL": dict(t=t_full_bl, display=f"t_full={t_full_bl:.2f} (test t={t_test_bl:.2f}, descriptive)"),
}

vt = ml.inference.evaluate(reg, stats)
with pd.option_context("display.max_colwidth", None):
    display(vt.set_index("item"))

## §10 TAKEAWAY — shrinkage conditions Σ but is not a source of return

- **Shrinkage conditions Σ a lot and moves weights little.** Median $\delta_{LW}$ 0.037 cuts the median condition number from 512.7 to 119.9; at the three snapshot dates LW moves GMV by 4.8–11.7% (½L1), MaxSharpe by 3.2–10.5% and BL(0.1) by 1.5–2.2%.
- **H1 (GMV) is supported under its registered rule, on the full window only.** GMV(LW) − GMV(S) is +0.25%/yr (t 2.34), test +0.17%/yr (t 0.94); not costs (gross +0.24%/yr), not risk (vol 3.17% → 3.19%), mostly $\Delta\alpha$ (+0.16%/yr). Plain-OLS t on three pairs; notebook 08 tests each run's Sharpe ratio and CAPM α with multiple-testing control.
- **H2 (MaxSharpe) is not supported; the sign is opposite.** +0.40%/yr (t 1.61); Sharpe 0.72 → 0.73 full, 1.13 → 1.05 test. BL(0.1): +0.07%/yr (t 1.00), the null holds. OAS agrees in sign for GMV and MaxSharpe, neither with t beyond ±2.
- **The UUP corner survives.** At 2022-12-31 GMV holds 55.35% UUP under LW (58.12% sample), MaxSharpe 74.09% (74.41%).
- **Against the benchmarks:** the best shrunk run, MaxSharpe(OAS), has a full-window Sharpe of 0.73, below 60/40's 0.91; GMV(LW) at 0.70 stays below EW's 0.74.

Shrinkage is a conditioning fix here, not a source of return; notebook 05 changes the objective instead (most diversified portfolio).